# Detonate the Maximum Bombs
You are given a list of bombs. The range of a bomb is defined as the area where its effect can be felt. This area is in the shape of a circle with the center as the location of the bomb.

The bombs are represented by a 0-indexed 2D integer array bombs where bombs[i] = [xi, yi, ri]. xi and yi denote the X-coordinate and Y-coordinate of the location of the ith bomb, whereas ri denotes the radius of its range.

You may choose to detonate a single bomb. When a bomb is detonated, it will detonate all bombs that lie in its range. These bombs will further detonate the bombs that lie in their ranges.

Given the list of bombs, return the maximum number of bombs that can be detonated if you are allowed to detonate only one bomb.

Input: bombs = [[2,1,3],[6,1,4]]\
Output: 2\
Explanation:\
The above figure shows the positions and ranges of the 2 bombs.
If we detonate the left bomb, the right bomb will not be affected.
But if we detonate the right bomb, both bombs will be detonated.
So the maximum bombs that can be detonated is max(1, 2) = 2.

Input: bombs = [[1,1,5],[10,10,5]]\
Output: 1\
Explanation:\
Detonating either bomb will not detonate the other bomb, so the maximum number of bombs that can be detonated is 1.

Input: bombs = [[1,2,3],[2,3,1],[3,4,2],[4,5,3],[5,6,4]]\
Output: 5\
Explanation:\
The best bomb to detonate is bomb 0 because:
- Bomb 0 detonates bombs 1 and 2. The red circle denotes the range of bomb 0.
- Bomb 2 detonates bomb 3. The blue circle denotes the range of bomb 2.
- Bomb 3 detonates bomb 4. The green circle denotes the range of bomb 3.\
Thus all 5 bombs are detonated.

Constraints:
* 1 <= bombs.length <= 100
* bombs[i].length == 3
* 1 <= xi, yi, ri <= $10^5$




# DFS on Adjacency List Graph

We can create a directed graph where bomb A is adjacent to B if A can detonate B. We determine this via the function `in_range(bomb1=A, bomb2=B)`. It calculates the distance between A and B, and returns True iff the distance is less than A's radius. It uses the distance formula $(A_x-B_x)^{2} + (A_y-B_y)^{2} = d^{2}$ where `d` is distance. Because Python's square root function can be imprecise, you should keep the comparison to $d^{2} <= r_{A}^{2}$ instead of trying to compare after taking the square roots. Otherwise, you may miss adjacent bombs.

We'll store adjacent bombs in an adjacency list, implemented as a hashmap. The hash-map is more straight forward for performing DFS, and we don't expect the graph to be densely connected. The key in the hash map should be the indices of each bomb in the list `bomb`. This is because multiple bombs can exist at the same coordinates, so we need a unique identifier to distinguish them in order to get the correct answer.

From there, we just need to do a DFS search starting at each bomb to find the maximum bombs we can detonate. Each time we start a new DFS search, we will need to reset the `seen` set.

## Analysis
* Time Complexity: O($N^3$)
    * Doing the DFS search for one bomb is O($N^2$), because at most there are $N^2$ edges to explore
    * Because we're looking for a maximum bomb detonations, we run DFS N times for each bomb which brings up the time complexity to O($N^3$)
* Space Complexity: O($N^2$)
    * the size of the adjacency list dominates at a size of O($N^2$)
    * the `seen` set only needs to store N bombs at most
    * the DFS stack depth doesn't go beyond O(N) because it's bounded by `seen` in its of the base cases.

In [ ]:
from typing import List


class Solution:
    def maximumDetonation(self, bombs: List[List[int]]) -> int:
        def in_range(bomb1, bomb2):
            if (bomb1[0]-bomb2[0])**2 + (bomb1[1]-bomb2[1])**2 <= bomb1[2]**2:
                return True
            return False
        
        adjacent = {i:list() for i in range(len(bombs))}

        for bomb1_i in range(len(bombs)):
            for bomb2_i in range(len(bombs)):
                if bomb1_i != bomb2_i and in_range(bombs[bomb1_i], bombs[bomb2_i]):
                    adjacent[bomb1_i].append(bomb2_i)
        
        seen = set()
        def explode_bomb(bomb_i):
            nonlocal seen
            nonlocal adjacent
            detonated = 1
            if bomb_i in seen:
                return 0
            seen.add(bomb_i)
            for bomb2_i in adjacent[bomb_i]:
                detonated += explode_bomb(bomb2_i)
            return detonated

        max_detonated = 0
        for bomb in range(len(bombs)):
            seen = set()
            max_detonated = max(max_detonated, explode_bomb(bomb))

        return max_detonated